# Latent time regimes vs flexible continuous time distributions (option C)

**Objective.** Test, falsifiably, whether the residual time heterogeneity reflects discrete latent time regimes or a continuous non-normal distribution: spline (GAM-type) location and variance, skew-normal, Student t and sinh-arcsinh residuals versus K = 2, 3 regimes with ordered intercepts and ability-dependent prevalence, all conditional on the same IRT response model. Null simulations, positive control, bootstrap stability, and an independent test on Forms B and C.

**Data.** Real mode needs the PISA 2025 public-use files processed as described in `data/README.md` (licensed by the OECD and never committed). For Colab, upload `data/processed/math_form*.parquet/.json` and `student_external.parquet` to the same relative paths, or mount Google Drive. Demo mode generates synthetic data. **Demo outputs are not findings.** The *saved results* cells display the PISA 2025 aggregate outputs committed in `results/`.

Errors are raised, never swallowed. Every output is written to `results/` with the form name; outputs from synthetic data are prefixed `demo_`.

In [ ]:
# --- Environment (Colab or local) -------------------------------------------
REPO_URL = ""   # GitHub repository; a private repository needs a token in the URL or an uploaded copy
DEMO = True            # True = synthetic demonstration (no PISA data needed); False = real PISA analysis
import os, sys, subprocess, platform
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("ability-adjusted-mcq-profiling"):
        if not REPO_URL:
            raise SystemExit("Set REPO_URL or upload the repository folder to /content.")
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("ability-adjusted-mcq-profiling")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.exists("configs") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")
sys.path.insert(0, "src"); sys.path.insert(0, "scripts")
import numpy, scipy, pandas, sklearn
print(platform.python_version(), "numpy", numpy.__version__, "scipy", scipy.__version__,
      "pandas", pandas.__version__, "sklearn", sklearn.__version__)
FORM = "demo" if DEMO else "math_formA"
print("Mode:", "SYNTHETIC DEMONSTRATION - not empirical results" if DEMO else "REAL PISA 2025 data")

In [ ]:
# 1. Synthetic demonstration: regimes are found when present and not when the distribution is merely skewed
import numpy as np
from scipy.special import expit
from scipy.stats import skewnorm
from mcq_profiles.latent_classes import ContinuousIRT
from mcq_profiles.temporal import ContinuousTimeModel, RegimeTimeModel, item_loglik_grid
rng = np.random.default_rng(0); n, J = 3000, 20
m = ContinuousIRT(dims=1, guessing=True); m.A = rng.uniform(0.8, 1.8, (J, 1)); m.d = rng.normal(0, 1, J); m.c = rng.uniform(.05, .2, J)
th = rng.normal(size=n); Y = (rng.random((n, J)) < m.c + (1 - m.c) * expit(th[:, None] * m.A[:, 0] + m.d)).astype(float)
LY = item_loglik_grid(m, Y); g = np.zeros(n, int); x = m.X[:, 0]
for name, v in [('true regimes', -0.3 * th + np.where(rng.random(n) < .3, 1.2, 0) + .3 * rng.normal(size=n)),
                ('skewed continuum', -0.3 * th + .5 * skewnorm.rvs(4, size=n, random_state=1))]:
    c = ContinuousTimeModel('shash', 'spline', 'spline').fit(LY, v, g, x)
    r = RegimeTimeModel(2, n_starts=3).fit(LY, v, g, x)
    print(f'{name:18s} BIC continuous {c.bic():8.1f}  BIC 2 regimes {r.bic():8.1f}  ->', 'regimes' if r.bic() < c.bic() else 'continuous')

In [ ]:
# 2. Real analysis (needs the PISA extracts): development, nulls, bootstrap, confirmation
import subprocess, sys
if not DEMO:
    for args in [['--part', 'dev'], ['--part', 'null'], ['--part', 'boot'], ['--part', 'confirm', '--form', 'math_formB'],
                 ['--part', 'confirm', '--form', 'math_formC']]:
        r = subprocess.run([sys.executable, 'scripts/run_temporal.py'] + args, capture_output=True, text=True)
        print(r.stdout[-3000:]); r.check_returncode()

In [ ]:
# 3. Saved PISA 2025 results
import json, pandas as pd
from pathlib import Path
from IPython.display import display, Markdown, Image
R = Path('results/temporal')
for f in sorted(R.glob('*.csv')):
    display(Markdown(f'**{f.name}**')); display(pd.read_csv(f))
for f in sorted(R.glob('*.json')):
    display(Markdown(f'**{f.name}**')); print(json.dumps(json.loads(f.read_text()), indent=1)[:3000])

In [ ]:
p = Path('results/figures/fig17_temporal_regimes.png')
display(Image(str(p))) if p.exists() else print('missing', p)

In [ ]:
p = Path('reports/temporal_regimes_report.md')
display(Markdown(p.read_text(encoding='utf8'))) if p.exists() else print('missing', p)